# Cross-Model Solver Test

Tests `CrossModelSolver` on a set of preset target words to see how often it lands on the correct answer.
Models are loaded once and the solver state is reset between runs. Each target is tested multiple times
(since random guesses introduce variance) to get a reliable success rate.

In [ ]:
import io
import contextlib
from semantle.semantle import Semantle
from solver import CrossModelSolver

# Load once — this takes a while (word2vec + sentence-transformer + encoding)
game = Semantle()
solver = CrossModelSolver(game)

In [ ]:
TARGET_WORDS = [
    "medical",
    "computer",
    "river",
    "music",
    "kitchen",
    "science",
    "happy",
    "animal",
    "weather",
    "sport",
]

In [ ]:
RUNS_PER_WORD = 5

def run_on_target(solver, target):
    """Reset solver state for a new target and run, returning (answer, correct, n_guesses, log)."""
    solver.semantle.word_of_the_day = target
    solver.guesses = []
    solver.candidates = list(range(len(solver.vocab)))
    solver.target_idx = solver.word_to_idx.get(target)

    log = io.StringIO()
    with contextlib.redirect_stdout(log):
        answer = solver.solve()

    n_guesses = len(solver.guesses)
    correct = (answer == target)
    return answer, correct, n_guesses, log.getvalue()


results = {}  # target -> list of (answer, correct, n_guesses)
for target in TARGET_WORDS:
    runs = []
    for run in range(RUNS_PER_WORD):
        answer, correct, n_guesses, log = run_on_target(solver, target)
        runs.append((answer, correct, n_guesses, log))
    results[target] = runs

    wins = sum(c for _, c, _, _ in runs)
    avg_guesses = sum(g for _, _, g, _ in runs) / len(runs)
    answers = [a for a, _, _, _ in runs]
    print(f"  {target:12s}  {wins}/{RUNS_PER_WORD} correct  avg_guesses={avg_guesses:.0f}  answers={answers}")

    # For correct runs, show first 6 and last 6 guesses
    for i, (a, c, g, log) in enumerate(runs):
        if not c:
            continue
        round_lines = [l for l in log.splitlines() if l.strip().startswith("Round")]
        first = round_lines[:6]
        last = round_lines[-6:]
        print(f"    --- run {i+1} ({g} guesses) ---")
        for line in first:
            print(f"    {line.strip()}")
        if len(round_lines) > 12:
            print(f"    ... ({len(round_lines) - 12} rounds omitted) ...")
        for line in last:
            if line not in first:
                print(f"    {line.strip()}")

In [ ]:
total_runs = sum(len(runs) for runs in results.values())
total_correct = sum(sum(c for _, c, _, _ in runs) for runs in results.values())
total_guesses = sum(sum(g for _, _, g, _ in runs) for runs in results.values())

print(f"Overall: {total_correct}/{total_runs} correct ({100*total_correct/total_runs:.0f}%)")
print(f"Average guesses: {total_guesses/total_runs:.0f}")